In [2]:
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 1.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 126.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 88.9 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [ ]:
import torch
import numpy as np
import open3d as o3d
import open3d.ml.torch as ml3d
import open3d.ml as o3dml

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(np.__version__)
print(o3d.__version__)

2.2.0+cu121
True
1.26.4
0.19.0


In [3]:
import numpy as np
import yaml
from pathlib import Path


class ScannetPrimitivesDet(o3dml.datasets.Scannet):

    def __init__(
        self,
        dataset_path: str,
        scannet_yaml_path: str,
        name: str = "ScannetPrimitivesDet",
        cache_dir: str = "./logs/cache",
        use_cache: bool = False,
        **kwargs,
    ):
        super().__init__(
            dataset_path=dataset_path,
            name=name,
            cache_dir=cache_dir,
            use_cache=use_cache,
            **kwargs,
        )

        self._scannet_yaml_path = scannet_yaml_path
        yaml_path = Path(scannet_yaml_path)
        if not yaml_path.exists():
            raise FileNotFoundError(f"Config file not found: {yaml_path}")

        with open(yaml_path, "r") as f:
            cfg = yaml.safe_load(f)

        self.classes = list(cfg["classes"])
        self.num_classes = len(self.classes)

        semantic_ids = cfg.get(
            "semantic_ids",
            list(range(len(self.classes))),
        )
        cat_ids = cfg.get(
            "cat_ids",
            list(range(len(self.classes))),
        )

        if len(semantic_ids) != self.num_classes:
            raise ValueError(
                f"Length of semantic_ids {len(semantic_ids)} "
                f"does not match number of classes {self.num_classes}"
            )

        if len(cat_ids) != self.num_classes:
            raise ValueError(
                f"Length of cat_ids {len(cat_ids)} "
                f"does not match number of classes {self.num_classes}"
            )

        self.semantic_ids = list(semantic_ids)
        self.cat_ids = np.array(cat_ids, dtype=np.int32)
        self.cat2label = {cat: i for i, cat in enumerate(self.classes)}

        ignored_label = cfg.get("ignored_label", -1)
        self.cat2label["ignored"] = ignored_label

        self.label2cat = {v: k for k, v in self.cat2label.items()}
        self.cat_ids2class = {
            raw_id: i for i, raw_id in enumerate(list(self.cat_ids))
        }

        self.label_to_names = self.get_label_to_names()

    def get_label_to_names(self):
        return self.label2cat

In [4]:
ds = ScannetPrimitivesDet(
    dataset_path="data/scannet_primitives",
    scannet_yaml_path="scannet_primitives.yaml"
)

In [5]:
split = ds.get_split("train")
sample = split.get_data(0)
points = sample["point"].astype(np.float32)
boxes = sample['bounding_boxes']
label = sample["label"].astype(np.int32)

print(f"Points shape: {points.shape}")
print(f"Labels shape: {label.shape}")

Points shape: (8192, 3)
Labels shape: (8192,)


In [6]:
names = ds.label_to_names
num_classes = int(max(names.keys())) + 1
rng = np.random.default_rng(0)
palette = rng.random((num_classes, 3))

In [7]:
import plotly.graph_objects as go


def plot_point_cloud(points, color=None, size=2, title="Point Cloud"):

    if isinstance(points, torch.Tensor):
        pts = points.detach().cpu().numpy()
    else:
        pts = np.asarray(points)

    if pts.ndim != 2 or (pts.shape[1] != 3 and pts.shape[0] != 3):
        raise ValueError(f"Ожидается массив формы (N,3) или (3,N), получено {pts.shape}")
    if pts.shape[1] != 3:
        pts = pts.T

    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]

    marker_kwargs = dict(size=size, opacity=0.9)
    if color is None:
        marker_kwargs["color"] = "royalblue"
    else:
        marker_kwargs["color"] = color

    scatter = go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=marker_kwargs
    )
    fig = go.Figure(data=[scatter])
    fig.update_layout(
        title=title,
        scene=dict(aspectmode='data'),
        margin=dict(l=0, r=0, t=30, b=0)
    )
    
    fig.show()

In [8]:
from open3d._ml3d.vis.boundingbox import BoundingBox3D

pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(points))
bbxs = [obj for obj in boxes]
ls = BoundingBox3D.create_lines(bbxs)
o3d.visualization.draw_geometries([pcd, ls])

[Open3D WARNING] GLFW Error: Wayland: The platform does not support setting the window position
[Open3D WARNING] Failed to initialize GLEW.
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.


In [10]:
cfg_file = 'pointpillars_primitives.yml'
cfg = o3dml.utils.Config.load_from_file(cfg_file)
model = ml3d.models.PointPillars(**cfg.model)

In [11]:
# CKPT = '/home/kitt/Projects/side_projects/TGU/det_task_20/pointpillars_kitti_202012221652utc.pth'

# ckpt = torch.load(CKPT, map_location="cpu")
# sd = ckpt.get("model_state_dict", ckpt)
# md = model.state_dict()

# loadable = {k:v for k,v in sd.items() if k in md and md[k].shape == v.shape}

# md.update(loadable)
# model.load_state_dict(md, strict=False)

In [12]:
pipeline = ml3d.pipelines.ObjectDetection(
    model=model,
    dataset=ds,
    **cfg.pipeline
)

In [13]:
points_xyz = sample["point"].astype(np.float32)   # (N, 3)
data_infer = {
    "point": points_xyz
    }

In [ ]:
res = pipeline.run_inference(data_infer)

In [16]:
pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(points_xyz))

bbxs = res[0][0:5]
ls = BoundingBox3D.create_lines(bbxs)
o3d.visualization.draw_geometries([pcd, ls], window_name="Pred")

In [17]:
# 1 epoch ~ 10 sec
pipeline.run_train()

training -  loss_cls: 0.484 loss_bbox: 1.781 loss_dir: 0.000 > loss: 2.265: 100%|██████████| 95/95 [00:08<00:00, 10.67it/s] 
validation: 100%|██████████| 142/142 [00:02<00:00, 55.89it/s]
training -  loss_cls: 0.277 loss_bbox: 0.955 loss_dir: 0.000 > loss: 1.232: 100%|██████████| 95/95 [00:08<00:00, 10.93it/s]
validation: 100%|██████████| 142/142 [00:02<00:00, 57.80it/s]
training -  loss_cls: 0.194 loss_bbox: 0.675 loss_dir: 0.000 > loss: 0.869: 100%|██████████| 95/95 [00:08<00:00, 10.95it/s]
validation: 100%|██████████| 142/142 [00:02<00:00, 57.01it/s]
training -  loss_cls: 0.176 loss_bbox: 0.728 loss_dir: 0.000 > loss: 0.904: 100%|██████████| 95/95 [00:08<00:00, 10.77it/s]
validation: 100%|██████████| 142/142 [00:02<00:00, 54.58it/s]
training -  loss_cls: 0.158 loss_bbox: 0.742 loss_dir: 0.000 > loss: 0.900: 100%|██████████| 95/95 [00:09<00:00, 10.56it/s]
validation: 100%|██████████| 142/142 [00:02<00:00, 58.74it/s]
training -  loss_cls: 0.126 loss_bbox: 0.583 loss_dir: 0.000 > loss: 

In [ ]:
pipeline.run_test()

validation: 100%|██████████| 4/4 [00:00<00:00, 12.81it/s]
testing: 0it [00:00, ?it/s]


In [ ]:
pipeline.valid_losses['mAP 3D']

0.7575757575757577

In [ ]:
pipeline.valid_losses

{'loss_cls': [array(1.3811221, dtype=float32),
  array(0.64252394, dtype=float32),
  array(0.4951614, dtype=float32),
  array(0.89057755, dtype=float32)],
 'loss_bbox': [array(1.7146034, dtype=float32),
  array(1.8904855, dtype=float32),
  array(2.0510824, dtype=float32),
  array(2.1632044, dtype=float32)],
 'loss_dir': [array(1.12805965e-05, dtype=float32),
  array(4.4674704e-05, dtype=float32),
  array(4.4103763e-05, dtype=float32),
  array(1.13975375e-05, dtype=float32)],
 'mAP BEV': 12.731947321982991,
 'mAP 3D': 0.7575757575757577}